# Module 1.0 — The Naked LLM

**Goal:** before we build any agent, let's demystify what a Large Language Model call actually *is*.

By the end of this notebook you will:
1. Call Ollama's `/api/chat` endpoint directly from Python — no libraries, no abstraction.
2. Inspect the JSON request/response structure.
3. Write a minimal `chat(messages, temperature) → str` function (~15 lines) that every subsequent module reuses.
4. Run the same Ising-model question at temperatures 0.1 / 1.0 / 2.0 and observe how output variance changes.
5. Maintain multi-turn context manually.

**Physics bridge — why this matters for us.** An LLM generates text by sampling tokens from a softmax distribution:

$$
P(\text{token}_i \mid \text{context}) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}
$$

That is *literally* a Boltzmann distribution over the vocabulary, with the "temperature" parameter `T` playing the role of $k_B T$ in statistical mechanics. The `temperature` slider in every chatbot UI is exactly the knob a condensed-matter physicist turns when studying a system at different thermal scales:
- $T \to 0$ → greedy / deterministic (ground-state dominated).
- $T = 1$ → default generation (finite-temperature equilibrium).
- $T \gg 1$ → high-entropy / nearly uniform sampling (disordered phase).

Keep that analogy in mind — we will return to it several times during the course.

> **Key insight.** An LLM is a *stateless conditional text generator*: `f(messages) → next_token_distribution`. Everything else we build from here — agents, RAG, tools, multi-agent systems — is scaffolding around this single operation.

---

### Prerequisites

- Ollama running locally (`ollama serve`, or the Ollama app on macOS).
- The `qwen3.5:4b` model pulled (`ollama pull qwen3.5:4b`).
- A Python ≥ 3.11 environment with `requests` installed.

If `setup/verify_setup.py` prints all-green, you are ready.


## 1. What are we actually calling?

Ollama exposes a REST API on `localhost:11434`. The one endpoint we need is:

```
POST /api/chat
```

The request body is a JSON document that looks like this:

```json
{
  "model":    "qwen3.5:4b",
  "messages": [
    {"role": "system", "content": "You are a helpful physics tutor."},
    {"role": "user",   "content": "What is the critical temperature of the 2D Ising model?"}
  ],
  "stream":   false,
  "options":  {"temperature": 0.7}
}
```

The response is another JSON document. The interesting field is `message.content` — the assistant's reply. Everything else is bookkeeping (token counts, timings, stop reason).

Let's make the call with nothing fancier than `requests.post` so you can see exactly what is on the wire.

In [3]:
import json
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL      = "llama3.1:8b"      # change to qwen3.5:4b or qwen3.5:9b if you have >=16 GB RAM

payload = {
    "model":    MODEL,
    "messages": [
        {"role": "system", "content": "You are a concise physics tutor."},
        {"role": "user",   "content": "What is the critical temperature of the 2D Ising model?"},
    ],
    "stream":   False,                   # get one JSON blob, not a token stream
    "options":  {"temperature": 0.7},
}

response = requests.post(OLLAMA_URL, json=payload, timeout=120)
response.raise_for_status()
data = response.json()

# Pretty-print the full response once, so you see every field.
print(json.dumps(data, indent=2)[:1200], "...")  # truncate for readability


{
  "model": "llama3.1:8b",
  "created_at": "2026-04-17T15:49:44.774408Z",
  "message": {
    "role": "assistant",
    "content": "The critical temperature (Tc) of the 2D Ising model is given by the Onsager solution: Tc = J/kB ln(1 + \u221a5), where J is the exchange energy and kB is Boltzmann's constant. This value is approximately 2.269 kJ/mol or, more conveniently for physicists, 2.27 in units of kT/J."
  },
  "done": true,
  "done_reason": "stop",
  "total_duration": 12183537042,
  "load_duration": 962945167,
  "prompt_eval_count": 36,
  "prompt_eval_duration": 7972750333,
  "eval_count": 82,
  "eval_duration": 3246296000
} ...


## 2. Dissecting the response

The fields that matter for us:

| Field | What it is |
|-------|-----------|
| `model` | which weights produced this reply |
| `message.role` | always `"assistant"` for us |
| `message.content` | the actual text — this is what we extract |
| `done` / `done_reason` | did the model finish naturally or hit a limit? |
| `prompt_eval_count`, `eval_count` | input / output tokens (useful for cost + speed) |
| `total_duration` (ns) | wall-clock time on the server |

Everything else we can ignore for now. The *entire* interface to the model is: **send a list of `{role, content}` messages, get one more `{role: assistant, content}` message back.**

In [4]:
# Just extract the reply. This single line is what a chat UI does under the hood.
print(data["message"]["content"])


The critical temperature (Tc) of the 2D Ising model is given by the Onsager solution: Tc = J/kB ln(1 + √5), where J is the exchange energy and kB is Boltzmann's constant. This value is approximately 2.269 kJ/mol or, more conveniently for physicists, 2.27 in units of kT/J.


## 3. A minimal `chat()` function

Let's wrap that in one small function. We will reuse it in **every** subsequent notebook — Module 1.1 wraps a reasoning loop around it, Module 1.2 injects retrieved passages into `messages`, and so on.

Design goals:
- Pure function: inputs → output, no globals.
- `temperature` is an explicit argument so we can experiment with it.
- Returns just the string reply (the metadata is useful for debugging but noisy for reuse).

In [5]:
def chat(messages: list[dict],
         model: str = MODEL,
         temperature: float = 0.7,
         url: str = OLLAMA_URL,
         timeout: int = 120) -> str:
    """One round-trip to the Ollama /api/chat endpoint.

    Args:
        messages: list of {'role': 'system'|'user'|'assistant', 'content': str}
        model: Ollama model tag
        temperature: softmax scaling of the token distribution (>=0)

    Returns:
        The assistant's reply as a plain string.
    """
    payload = {
        "model":    model,
        "messages": messages,
        "stream":   False,
        "options":  {"temperature": temperature},
    }
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["message"]["content"]


# Smoke test
reply = chat(
    messages=[
        {"role": "system", "content": "You are a concise physics tutor."},
        {"role": "user",   "content": "In one sentence, what is the 2D Ising model?"},
    ],
    temperature=0.3,
)
print(reply)


The 2D Ising model is a statistical mechanics model that describes the behavior of magnetic dipoles on a square lattice, with nearest-neighbor interactions and temperature-dependent phase transitions between ordered (ferromagnetic) and disordered (paramagnetic) states.


## 4. Temperature and the Boltzmann parallel

Time for the first experiment. We will ask the **same** question five times, at three different temperatures, and compare:

- **`temperature = 0.1`** — the distribution is sharply peaked on the most likely token. Replies should be near-identical across the 5 runs. Analogous to a frozen, ordered phase.
- **`temperature = 1.0`** — default generation. Some variation across runs but the content stays on topic.
- **`temperature = 2.0`** — strongly broadened distribution, rare tokens become plausible. Expect higher variance and occasional hallucination.

The **content** of our prompt is deliberately a question whose correct answer is a single number — $T_c = 2 / \ln(1+\sqrt{2}) \approx 2.269\,J/k_B$ — so it is easy to spot when the model slips.

In [6]:
import textwrap

QUESTION = (
    "What is the exact critical temperature T_c of the 2D Ising model "
    "on a square lattice with nearest-neighbour ferromagnetic coupling J? "
    "Give the formula and the numerical value in units of J/k_B. Be concise."
)

SYSTEM = "You are a concise physics tutor. Answer in at most 2 sentences."

def sample_runs(temperature: float, n: int = 3) -> list[str]:
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": QUESTION}]
    return [chat(msgs, temperature=temperature) for _ in range(n)]

for T in (0.1, 1.0, 2.0):
    print(f"\n===== temperature = {T}  (n = 3 samples) =====")
    for i, reply in enumerate(sample_runs(T, n=3), start=1):
        # Wrap long lines so they fit in a notebook cell.
        wrapped = textwrap.fill(reply.strip(), width=88, subsequent_indent="    ")
        print(f"  [{i}] {wrapped}\n")



===== temperature = 0.1  (n = 3 samples) =====
  [1] The exact critical temperature is given by T_c = (2J/k_B) \* ln(1 + √2). The numerical
    value is approximately 2.269.

  [2] The exact critical temperature for the 2D Ising model on a square lattice is given by
    T_c = (2J/k_B) \* ln(1 + √2). This evaluates to approximately 2.269 k_B/J.

  [3] The exact critical temperature for the 2D Ising model on a square lattice is given by
    T_c = (2J/k_B) \* ln(1 + √2). This evaluates to approximately 2.269 k_B/J.


===== temperature = 1.0  (n = 3 samples) =====
  [1] The exact critical temperature for the 2D Ising model is given by T_c = (2/J) * ln(1 +
    sqrt(2)). Numerically, this evaluates to approximately 2.269 k_B / J.

  [2] The exact critical temperature of the 2D Ising model is given by T_c = (2 J / k_B) ln(1
    + √5). In units of J/k_B, this evaluates to T_c ≈ 2.269.

  [3] The exact critical temperature for the 2D Ising model is given by T_c = (2J / k_B) *
    ln(1 + √5). T

**What to look for in the output above.** Read all 9 replies carefully and answer for yourself:

1. At $T = 0.1$: are the 3 samples almost identical? They should be — small `temperature` ≈ deterministic decoding.
2. At $T = 1.0$: do they differ in wording but agree on the numerical value $2.269$?
3. At $T = 2.0$: does any reply show degraded quality — wrong formula, wrong value, drift into unrelated physics, or outright gibberish?

This is the *same* signature a physicist sees in a Monte-Carlo simulation of a spin system: low-$T$ runs produce nearly identical configurations, high-$T$ runs produce configurations with very different magnetisation. This parallel recurs in the MAS Part: agent-debate and consensus dynamics admit the same temperature-driven framing, and the statistical-mechanics intuition is the right tool for reasoning about them.

## 5. Multi-turn conversation — state is your responsibility

The `/api/chat` endpoint is **stateless**. If you want the model to remember turn 1 when you ask turn 2, you must send the full history yourself. The "conversation" is just a growing `messages` list.

Below: a two-turn exchange. Watch how the second user message is contextually meaningful only because the assistant's previous answer is included in the messages list.

In [7]:
history: list[dict] = [
    {"role": "system",
     "content": "You are a concise physics tutor. Use at most 3 sentences."},
    {"role": "user",
     "content": "What is the critical temperature of the 2D Ising model?"},
]

# --- Turn 1 -----------------------------------------------------------
reply_1 = chat(history, temperature=0.2)
history.append({"role": "assistant", "content": reply_1})
print("ASSISTANT (turn 1):")
print(textwrap.fill(reply_1.strip(), width=88), "\n")

# --- Turn 2: a follow-up that only makes sense in context -------------
history.append({"role": "user",
                "content": "How was this value derived, and by whom?"})
reply_2 = chat(history, temperature=0.2)
history.append({"role": "assistant", "content": reply_2})
print("ASSISTANT (turn 2):")
print(textwrap.fill(reply_2.strip(), width=88), "\n")

# The full history now has 5 messages: system, user, assistant, user, assistant.
print(f"history now has {len(history)} messages.")


ASSISTANT (turn 1):
The 2D Ising model has no phase transition, meaning it does not have a critical
temperature above which spontaneous magnetization occurs. This is due to the fact that
the correlation length diverges at zero temperature in two dimensions. The model
exhibits a continuous phase transition only at zero temperature. 

ASSISTANT (turn 2):
The absence of a phase transition in the 2D Ising model was first rigorously proven by
Lars Onsager in 1944 using a solution to the two-point correlation function. He showed
that the free energy is analytic at all temperatures, implying no phase transition. This
result has since been generalized and rederived using various methods, including
conformal field theory and lattice gauge theory. 

history now has 5 messages.


**The crucial observation.** The model appears to "remember" turn 1 only because we manually appended the assistant's reply to `history` before sending turn 2. If you had sent just the second user message on its own, the model would have no idea what "this value" refers to.

Every agent framework we touch this week (smolagents, CrewAI, …) is, at its core, a system for managing this `messages` list on your behalf. They decide *which* past messages to keep, *which* tool results to inject, and *how* to format them. Knowing what the list actually looks like makes those frameworks transparent instead of magic.

## 6. Exercises

Take 5 minutes. These are quick and reinforce the core insight.

**E1 (temperature sweep).** Pick a physics question with a numeric answer (e.g. "What is the fine-structure constant?" or "What is the Schwarzschild radius of the Sun in km?") and run it 5 times at $T \in \{0.0, 0.5, 1.0, 1.5, 2.0\}$. Count how many of the 5 replies give the same numeric answer at each temperature. Plot if you feel like it.

**E2 (hallucination probing).** Ask the model about a *fictitious* physics concept, e.g. *"What is the Lindqvist-Tanaka coupling constant in lattice QCD?"*. At low $T$ it will often make something up confidently; at high $T$ it may ramble. Observe: **the LLM has no built-in uncertainty signal.** Module 1.2 (RAG) and Module 1.4 (Reflection) are the two mechanisms we will use to fix that.

**E3 (system prompt sensitivity).** Keep the user question identical and change only the system prompt from `"You are a concise physics tutor."` to `"You are a skeptical PhD examiner. Challenge the student if the question is ill-posed."`. How does the reply change? This is a preview of the *role design* that drives the multi-agent system in the MAS Part.

In [ ]:
# --- E1 scaffolding: fill in a question, run the loop, compare -------
QUESTION_E1 = "YOUR QUESTION HERE"

# for T in (0.0, 0.5, 1.0, 1.5, 2.0):
#     replies = sample_runs_... (you can copy/adapt sample_runs above)
#     print(T, replies)


## Key takeaway

> **An LLM is a conditional text generator.** Everything we build from here — tool use, retrieval, reflection, multi-agent systems — is scaffolding around a single operation: `f(messages, temperature) → reply`.

### What we built in this notebook
- `chat(messages, temperature, model) → str` — our reusable LLM primitive.
- A concrete understanding that the "conversation" is just a list we maintain.
- A first intuition for the `temperature` parameter as a softmax scaling, mirroring the Boltzmann weight $e^{-E/k_B T}$.

### What Module 1.1 adds
Nothing replaces the `chat()` function you wrote above. Module 1.1 wraps a **loop** around it:

```
while not done:
    reply = chat(history, temperature=...)
    if reply contains an "Action:" block:
        run the tool, append the Observation to history
    else:
        done = True
```

That loop + two tools (calculator + unit converter) is a complete **ReAct agent**. One loop. Two tools. That is all an agent is.
